<a href="https://colab.research.google.com/github/nanomelk/Analista-de-Datos-II/blob/mariano/ABP2_Home_Credit_Default_Risk.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **EVIDENCIA 1 - Comprensión del Negocio - Datos - Análisis exploratorio**

# **1. COMPRENSIÓN DEL PROBLEMA DE NEGOCIO**

### **Contexto Financiero y Necesidad**
En el sector crediticio, la evaluación precisa del riesgo es fundamental para la sostenibilidad y rentabilidad de la entidad. Otorgar créditos a clientes que posteriormente incurren en mora genera pérdidas financieras directas por capital no recuperado y eleva los costos de cobranza. Por otro lado, mantener un criterio de aprobación extremadamente rígido provoca el rechazo de clientes solventes, reduciendo los ingresos por intereses. La entidad financiera necesita optimizar la toma de decisiones para maximizar la aprobación de créditos rentables y controlar la tasa de morosidad.

---

### **Objetivo del Proyecto**
El objetivo principal de esta Prueba de Concepto (PoC) es desarrollar y evaluar un modelo predictivo (Regresión Logística) capaz de estimar la probabilidad de incumplimiento de pago (*default*) de los solicitantes de crédito a partir del archivo `application_train.csv`.

* **`TARGET = 0`**: Cliente que cumple correctamente sus pagos.
* **`TARGET = 1`**: Cliente que presenta incumplimiento o *default*.

---

### **Valor del Machine Learning para el Negocio**
La implementación de una solución basada en aprendizaje automático aporta un valor estratégico directo a la gestión del riesgo crediticio:

1. **Optimización de la Cartera y Rentabilidad:**
   * Minimiza el riesgo de morosidad al identificar patrones sutiles de incumplimiento.
   * Reduce el costo de oportunidad al evitar el rechazo injustificado de solicitantes aptos.

2. **Automatización y Consistencia:**
   * Permite evaluar solicitudes de forma estandarizada, rápida y sin sesgos subjetivos.

3. **Detección de Patrones Complejos:**
   * Identifica interacciones entre múltiples variables (socioeconómicas, demográficas y de scoring externo) que las reglas tradicionales basadas en umbrales fijos no capturan.

4. **Estrategia de Riesgo-Precio (*Risk-Based Pricing*):**
   * Al estimar la probabilidad continua de default de cada solicitante, la entidad puede graduar los montos de crédito, ajustar las tasas de interés o solicitar garantías adicionales según el perfil de riesgo del cliente.

---



### **Pregunta de Negocio**
> **¿Cómo podemos predecir con precisión la probabilidad de incumplimiento de pago (*default*) de los solicitantes de crédito a partir de su perfil socioeconómico y financiero, para optimizar las aprobaciones y reducir la tasa de morosidad sin impactar el crecimiento de la cartera?**

In [1]:
# Montar Google Drive en Colab
#from google.colab import drive
#drive.mount('/content/drive')

# Importar las librerías principales de Ciencia de Datos
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Configuración visual básica
%matplotlib inline
pd.set_option('display.max_columns', None)


In [19]:
# Carga del archivo directamente desde el Drive

#HomeCredit_CD = pd.read_csv('/content/drive/MyDrive/application_train.csv')
#ruta = r"https://drive.google.com/uc?export=download&id=1x9151g2ZndSt87zZ03MEdBd6ak3rY5aG"
import gdown
import pandas as pd

# 1. Descargar el archivo al disco local de Colab
url = "https://drive.google.com/uc?export=download&id=1x9151g2ZndSt87zZ03MEdBd6ak3rY5aG"
gdown.download(url, 'application_train.csv', quiet=False)

# 2. Leer el CSV descargado
HomeCredit_CD = pd.read_csv('application_train.csv')

# 3. Mostrar las primeras filas
display(HomeCredit_CD.head())
#HomeCredit_CD = pd.read_csv(ruta)
display(HomeCredit_CD.head())

Downloading...
From (original): https://drive.google.com/uc?export=download&id=1x9151g2ZndSt87zZ03MEdBd6ak3rY5aG
From (redirected): https://drive.google.com/uc?export=download&id=1x9151g2ZndSt87zZ03MEdBd6ak3rY5aG&confirm=t&uuid=ee5fe545-24f0-4333-8f25-95d8c75824e2
To: /content/application_train.csv
100%|██████████| 166M/166M [00:02<00:00, 74.6MB/s]


,SK_ID_CURR,TARGET,NAME_CONTRACT_TYPE,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,AMT_CREDIT,AMT_ANNUITY,AMT_GOODS_PRICE,NAME_TYPE_SUITE,NAME_INCOME_TYPE,NAME_EDUCATION_TYPE,NAME_FAMILY_STATUS,NAME_HOUSING_TYPE,REGION_POPULATION_RELATIVE,DAYS_BIRTH,DAYS_EMPLOYED,DAYS_REGISTRATION,DAYS_ID_PUBLISH,OWN_CAR_AGE,FLAG_MOBIL,FLAG_EMP_PHONE,FLAG_WORK_PHONE,FLAG_CONT_MOBILE,FLAG_PHONE,FLAG_EMAIL,OCCUPATION_TYPE,CNT_FAM_MEMBERS,REGION_RATING_CLIENT,REGION_RATING_CLIENT_W_CITY,WEEKDAY_APPR_PROCESS_START,HOUR_APPR_PROCESS_START,REG_REGION_NOT_LIVE_REGION,REG_REGION_NOT_WORK_REGION,LIVE_REGION_NOT_WORK_REGION,REG_CITY_NOT_LIVE_CITY,REG_CITY_NOT_WORK_CITY,LIVE_CITY_NOT_WORK_CITY,ORGANIZATION_TYPE,EXT_SOURCE_1,EXT_SOURCE_2,EXT_SOURCE_3,APARTMENTS_AVG,BASEMENTAREA_AVG,YEARS_BEGINEXPLUATATION_AVG,YEARS_BUILD_AVG,COMMONAREA_AVG,ELEVATORS_AVG,ENTRANCES_AVG,FLOORSMAX_AVG,FLOORSMIN_AVG,LANDAREA_AVG,LIVINGAPARTMENTS_AVG,LIVINGAREA_AVG,NONLIVINGAPARTMENTS_AVG,NONLIVINGAREA_AVG,APARTMENTS_MODE,BASEMENTAREA_MODE,YEARS_BEGINEXPLUATATION_MODE,YEARS_BUILD_MODE,COMMONAREA_MODE,ELEVATORS_MODE,ENTRANCES_MODE,FLOORSMAX_MODE,FLOORSMIN_MODE,LANDAREA_MODE,LIVINGAPARTMENTS_MODE,LIVINGAREA_MODE,NONLIVINGAPARTMENTS_MODE,NONLIVINGAREA_MODE,APARTMENTS_MEDI,BASEMENTAREA_MEDI,YEARS_BEGINEXPLUATATION_MEDI,YEARS_BUILD_MEDI,COMMONAREA_MEDI,ELEVATORS_MEDI,ENTRANCES_MEDI,FLOORSMAX_MEDI,FLOORSMIN_MEDI,LANDAREA_MEDI,LIVINGAPARTMENTS_MEDI,LIVINGAREA_MEDI,NONLIVINGAPARTMENTS_MEDI,NONLIVINGAREA_MEDI,FONDKAPREMONT_MODE,HOUSETYPE_MODE,TOTALAREA_MODE,WALLSMATERIAL_MODE,EMERGENCYSTATE_MODE,OBS_30_CNT_SOCIAL_CIRCLE,DEF_30_CNT_SOCIAL_CIRCLE,OBS_60_CNT_SOCIAL_CIRCLE,DEF_60_CNT_SOCIAL_CIRCLE,DAYS_LAST_PHONE_CHANGE,FLAG_DOCUMENT_2,FLAG_DOCUMENT_3,FLAG_DOCUMENT_4,FLAG_DOCUMENT_5,FLAG_DOCUMENT_6,FLAG_DOCUMENT_7,FLAG_DOCUMENT_8,FLAG_DOCUMENT_9,FLAG_DOCUMENT_10,FLAG_DOCUMENT_11,FLAG_DOCUMENT_12,FLAG_DOCUMENT_13,FLAG_DOCUMENT_14,FLAG_DOCUMENT_15,FLAG_DOCUMENT_16,FLAG_DOCUMENT_17,FLAG_DOCUMENT_18,FLAG_DOCUMENT_19,FLAG_DOCUMENT_20,FLAG_DOCUMENT_21,AMT_REQ_CREDIT_BUREAU_HOUR,AMT_REQ_CREDIT_BUREAU_DAY,AMT_REQ_CREDIT_BUREAU_WEEK,AMT_REQ_CREDIT_BUREAU_MON,AMT_REQ_CREDIT_BUREAU_QRT,AMT_REQ_CREDIT_BUREAU_YEAR
0,100002,1,Cash loans,M,N,Y,0,202500.0,406597.5,24700.5,351000.0,Unaccompanied,Working,Secondary / secondary special,Single / not married,House / apartment,0.018801,-9461,-637,-3648.0,-2120,NaN,1,1,0,1,1,0,Laborers,1.0,2,2,WEDNESDAY,10,0,0,0,0,0,0,Business Entity Type 3,0.083037,0.262949,0.139376,0.0247,0.0369,0.9722,0.6192,0.0143,0.00,0.0690,0.0833,0.1250,0.0369,0.0202,0.0190,0.0000,0.0000,0.0252,0.0383,0.9722,0.6341,0.0144,0.0000,0.0690,0.0833,0.1250,0.0377,0.022,0.0198,0.0,0.0,0.0250,0.0369,0.9722,0.6243,0.0144,0.00,0.0690,0.0833,0.1250,0.0375,0.0205,0.0193,0.0000,0.00,reg oper account,block of flats,0.0149,"Stone, brick",No,2.0,2.0,2.0,2.0,-1134.0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0.0,0.0,0.0,0.0,0.0,1.0
1,100003,0,Cash loans,F,N,N,0,270000.0,1293502.5,35698.5,1129500.0,Family,State servant,Higher education,Married,House / apartment,0.003541,-16765,-1188,-1186.0,-291,NaN,1,1,0,1,1,0,Core staff,2.0,1,1,MONDAY,11,0,0,0,0,0,0,School,0.311267,0.622246,NaN,0.0959,0.0529,0.9851,0.7960,0.0605,0.08,0.0345,0.2917,0.3333,0.0130,0.0773,0.0549,0.0039,0.0098,0.0924,0.0538,0.9851,0.8040,0.0497,0.0806,0.0345,0.2917,0.3333,0.0128,0.079,0.0554,0.0,0.0,0.0968,0.0529,0.9851,0.7987,0.0608,0.08,0.0345,0.2917,0.3333,0.0132,0.0787,0.0558,0.0039,0.01,reg oper account,block of flats,0.0714,Block,No,1.0,0.0,1.0,0.0,-828.0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0
2,100004,0,Revolving loans,M,Y,Y,0,67500.0,135000.0,6750.0,135000.0,Unaccompanied,Working,Secondary / secondary special,Single / not married,House / apartment,0.010032,-19046,-225,-4260.0,-2531,26.0,1,1,1,1,1,0,Laborers,1.0,2,2,MONDAY,9,0,0,0,0,0,0,Government,NaN,0.555912,0.729567,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,

,SK_ID_CURR,TARGET,NAME_CONTRACT_TYPE,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,AMT_CREDIT,AMT_ANNUITY,AMT_GOODS_PRICE,NAME_TYPE_SUITE,NAME_INCOME_TYPE,NAME_EDUCATION_TYPE,NAME_FAMILY_STATUS,NAME_HOUSING_TYPE,REGION_POPULATION_RELATIVE,DAYS_BIRTH,DAYS_EMPLOYED,DAYS_REGISTRATION,DAYS_ID_PUBLISH,OWN_CAR_AGE,FLAG_MOBIL,FLAG_EMP_PHONE,FLAG_WORK_PHONE,FLAG_CONT_MOBILE,FLAG_PHONE,FLAG_EMAIL,OCCUPATION_TYPE,CNT_FAM_MEMBERS,REGION_RATING_CLIENT,REGION_RATING_CLIENT_W_CITY,WEEKDAY_APPR_PROCESS_START,HOUR_APPR_PROCESS_START,REG_REGION_NOT_LIVE_REGION,REG_REGION_NOT_WORK_REGION,LIVE_REGION_NOT_WORK_REGION,REG_CITY_NOT_LIVE_CITY,REG_CITY_NOT_WORK_CITY,LIVE_CITY_NOT_WORK_CITY,ORGANIZATION_TYPE,EXT_SOURCE_1,EXT_SOURCE_2,EXT_SOURCE_3,APARTMENTS_AVG,BASEMENTAREA_AVG,YEARS_BEGINEXPLUATATION_AVG,YEARS_BUILD_AVG,COMMONAREA_AVG,ELEVATORS_AVG,ENTRANCES_AVG,FLOORSMAX_AVG,FLOORSMIN_AVG,LANDAREA_AVG,LIVINGAPARTMENTS_AVG,LIVINGAREA_AVG,NONLIVINGAPARTMENTS_AVG,NONLIVINGAREA_AVG,APARTMENTS_MODE,BASEMENTAREA_MODE,YEARS_BEGINEXPLUATATION_MODE,YEARS_BUILD_MODE,COMMONAREA_MODE,ELEVATORS_MODE,ENTRANCES_MODE,FLOORSMAX_MODE,FLOORSMIN_MODE,LANDAREA_MODE,LIVINGAPARTMENTS_MODE,LIVINGAREA_MODE,NONLIVINGAPARTMENTS_MODE,NONLIVINGAREA_MODE,APARTMENTS_MEDI,BASEMENTAREA_MEDI,YEARS_BEGINEXPLUATATION_MEDI,YEARS_BUILD_MEDI,COMMONAREA_MEDI,ELEVATORS_MEDI,ENTRANCES_MEDI,FLOORSMAX_MEDI,FLOORSMIN_MEDI,LANDAREA_MEDI,LIVINGAPARTMENTS_MEDI,LIVINGAREA_MEDI,NONLIVINGAPARTMENTS_MEDI,NONLIVINGAREA_MEDI,FONDKAPREMONT_MODE,HOUSETYPE_MODE,TOTALAREA_MODE,WALLSMATERIAL_MODE,EMERGENCYSTATE_MODE,OBS_30_CNT_SOCIAL_CIRCLE,DEF_30_CNT_SOCIAL_CIRCLE,OBS_60_CNT_SOCIAL_CIRCLE,DEF_60_CNT_SOCIAL_CIRCLE,DAYS_LAST_PHONE_CHANGE,FLAG_DOCUMENT_2,FLAG_DOCUMENT_3,FLAG_DOCUMENT_4,FLAG_DOCUMENT_5,FLAG_DOCUMENT_6,FLAG_DOCUMENT_7,FLAG_DOCUMENT_8,FLAG_DOCUMENT_9,FLAG_DOCUMENT_10,FLAG_DOCUMENT_11,FLAG_DOCUMENT_12,FLAG_DOCUMENT_13,FLAG_DOCUMENT_14,FLAG_DOCUMENT_15,FLAG_DOCUMENT_16,FLAG_DOCUMENT_17,FLAG_DOCUMENT_18,FLAG_DOCUMENT_19,FLAG_DOCUMENT_20,FLAG_DOCUMENT_21,AMT_REQ_CREDIT_BUREAU_HOUR,AMT_REQ_CREDIT_BUREAU_DAY,AMT_REQ_CREDIT_BUREAU_WEEK,AMT_REQ_CREDIT_BUREAU_MON,AMT_REQ_CREDIT_BUREAU_QRT,AMT_REQ_CREDIT_BUREAU_YEAR
0,100002,1,Cash loans,M,N,Y,0,202500.0,406597.5,24700.5,351000.0,Unaccompanied,Working,Secondary / secondary special,Single / not married,House / apartment,0.018801,-9461,-637,-3648.0,-2120,NaN,1,1,0,1,1,0,Laborers,1.0,2,2,WEDNESDAY,10,0,0,0,0,0,0,Business Entity Type 3,0.083037,0.262949,0.139376,0.0247,0.0369,0.9722,0.6192,0.0143,0.00,0.0690,0.0833,0.1250,0.0369,0.0202,0.0190,0.0000,0.0000,0.0252,0.0383,0.9722,0.6341,0.0144,0.0000,0.0690,0.0833,0.1250,0.0377,0.022,0.0198,0.0,0.0,0.0250,0.0369,0.9722,0.6243,0.0144,0.00,0.0690,0.0833,0.1250,0.0375,0.0205,0.0193,0.0000,0.00,reg oper account,block of flats,0.0149,"Stone, brick",No,2.0,2.0,2.0,2.0,-1134.0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0.0,0.0,0.0,0.0,0.0,1.0
1,100003,0,Cash loans,F,N,N,0,270000.0,1293502.5,35698.5,1129500.0,Family,State servant,Higher education,Married,House / apartment,0.003541,-16765,-1188,-1186.0,-291,NaN,1,1,0,1,1,0,Core staff,2.0,1,1,MONDAY,11,0,0,0,0,0,0,School,0.311267,0.622246,NaN,0.0959,0.0529,0.9851,0.7960,0.0605,0.08,0.0345,0.2917,0.3333,0.0130,0.0773,0.0549,0.0039,0.0098,0.0924,0.0538,0.9851,0.8040,0.0497,0.0806,0.0345,0.2917,0.3333,0.0128,0.079,0.0554,0.0,0.0,0.0968,0.0529,0.9851,0.7987,0.0608,0.08,0.0345,0.2917,0.3333,0.0132,0.0787,0.0558,0.0039,0.01,reg oper account,block of flats,0.0714,Block,No,1.0,0.0,1.0,0.0,-828.0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0
2,100004,0,Revolving loans,M,Y,Y,0,67500.0,135000.0,6750.0,135000.0,Unaccompanied,Working,Secondary / secondary special,Single / not married,House / apartment,0.010032,-19046,-225,-4260.0,-2531,26.0,1,1,1,1,1,0,Laborers,1.0,2,2,MONDAY,9,0,0,0,0,0,0,Government,NaN,0.555912,0.729567,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,

In [3]:
#import os
"""from google.colab import auth
from pydrive2.auth import GoogleAuth
from pydrive2.drive import GoogleDrive
from oauth2client.client import GoogleCredentials
import zipfile

# 1. Autenticar tu sesión de Google
auth.authenticate_user()
gauth = GoogleAuth()
gauth.credentials = GoogleCredentials.get_application_default()
drive = GoogleDrive(gauth)

# 2. Buscar el archivo zip dentro de tus compartidos
file_list = drive.ListFile({'q': "title = 'home-credit-default-risk.zip' and trashed = false"}).GetList()

if file_list:
    zip_id = file_list[0]['id']
    print(f"Descargando archivo ZIP (ID: {zip_id})...")

    zip_file = drive.CreateFile({'id': zip_id})
    zip_file.GetContentFile('home-credit-default-risk.zip')

    # 3. Descomprimir todos los CSVs en el directorio actual
    print("Descomprimiendo archivos...")
    with zipfile.ZipFile('home-credit-default-risk.zip', 'r') as zip_ref:
        zip_ref.extractall('.')

    print("¡Listo! Archivos descomprimidos:")
    print([f for f in os.listdir('.') if f.endswith('.csv')])
else:
    print("No se encontró el archivo zip en Compartidos conmigo.")"""

'from google.colab import auth\nfrom pydrive2.auth import GoogleAuth\nfrom pydrive2.drive import GoogleDrive\nfrom oauth2client.client import GoogleCredentials\nimport zipfile\n\n# 1. Autenticar tu sesión de Google\nauth.authenticate_user()\ngauth = GoogleAuth()\ngauth.credentials = GoogleCredentials.get_application_default()\ndrive = GoogleDrive(gauth)\n\n# 2. Buscar el archivo zip dentro de tus compartidos\nfile_list = drive.ListFile({\'q\': "title = \'home-credit-default-risk.zip\' and trashed = false"}).GetList()\n\nif file_list:\n    zip_id = file_list[0][\'id\']\n    print(f"Descargando archivo ZIP (ID: {zip_id})...")\n\n    zip_file = drive.CreateFile({\'id\': zip_id})\n    zip_file.GetContentFile(\'home-credit-default-risk.zip\')\n\n    # 3. Descomprimir todos los CSVs en el directorio actual\n    print("Descomprimiendo archivos...")\n    with zipfile.ZipFile(\'home-credit-default-risk.zip\', \'r\') as zip_ref:\n        zip_ref.extractall(\'.\')\n\n    print("¡Listo! Archivos d

In [4]:
#from google.colab import drive
#drive.mount('/content/drive')

In [15]:
HomeCredit_CD.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 0 entries
Data columns (total 3 columns):
 #   Column                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                 

Volumen de Datos:

*   307,511 registros (filas que representan a cada solicitante de crédito)

*   122 características (columnas con atributos del cliente y la solicitud).




In [6]:
# visualizacion de los 3 primeros registros
HomeCredit_CD.head(3)

,"<!DOCTYPE html><html><head><title>Google Drive - Virus scan warning</title><meta http-equiv=""content-type"" content=""text/html; charset=utf-8""/><style nonce=""0DJv_iFzrWvECY2jJuN1Gg"">.goog-link-button{position:relative;color:#15c;text-decoration:underline;cursor:pointer}.goog-link-button-disabled{color:#ccc;text-decoration:none;cursor:default}body{color:#222;font:normal 13px/1.4 arial",sans-serif;margin:0}.grecaptcha-badge{visibility:hidden}.uc-main{padding-top:50px;text-align:center}#uc-dl-icon{display:inline-block;margin-top:16px;padding-right:1em;vertical-align:top}#uc-text{display:inline-block;max-width:68ex;text-align:left}.uc-error-caption,".uc-warning-caption{color:#222;font-size:16px}#uc-download-link{text-decoration:none}.uc-name-size a{color:#15c;text-decoration:none}.uc-name-size a:visited{color:#61c;text-decoration:none}.uc-name-size a:active{color:#d14836;text-decoration:none}.uc-footer{color:#777;font-size:11px;padding-bottom:5ex;padding-top:5ex;text-align:center}.uc-footer a{color:#15c}.uc-footer a:visited{color:#61c}.uc-footer a:active{color:#d14836}.uc-footer-divider{color:#ccc;width:100%}.goog-inline-block{position:relative;display:-moz-inline-box;display:inline-block}* html .goog-inline-block{display:inline}:first-child+html .goog-inline-block{display:inline}sentinel{}</style><link rel=""icon"" href=""//ssl.gstatic.com/docs/doclist/images/drive_favicon_2026_32dp.png""/></head><body><div class=""uc-main""><div id=""uc-dl-icon"" class=""image-container""><div class=""drive-sprite-aux-download-file""></div></div><div id=""uc-text""><p class=""uc-warning-caption"">Google Drive can't scan this file for viruses.</p><p class=""uc-warning-subcaption""><span class=""uc-name-size""><a href=""/open?id=1x9151g2ZndSt87zZ03MEdBd6ak3rY5aG"">application_train.csv</a> (158M)</span> is too large for Google to scan for viruses. Would you still like to download this file?</p><form id=""download-form"" action=""https://drive.usercontent.google.com/download"" method=""get""><input type=""submit"" id=""uc-download-link"" class=""goog-inline-block jfk-button jfk-button-action"" value=""Download anyway""/><input type=""hidden"" name=""id"" value=""1x9151g2ZndSt87zZ03MEdBd6ak3rY5aG""><input type=""hidden"" name=""export"" value=""download""><input type=""hidden"" name=""confirm"" value=""t""><input type=""hidden"" name=""uuid"" value=""79faa669-e8af-463f-8683-474e69f2a098""></form></div></div><div class=""uc-footer""><hr class=""uc-footer-divider""></div></body></html>"


In [7]:
# visualizacion de datos estadisticos de las variables
HomeCredit_CD.describe()

,"<!DOCTYPE html><html><head><title>Google Drive - Virus scan warning</title><meta http-equiv=""content-type"" content=""text/html; charset=utf-8""/><style nonce=""0DJv_iFzrWvECY2jJuN1Gg"">.goog-link-button{position:relative;color:#15c;text-decoration:underline;cursor:pointer}.goog-link-button-disabled{color:#ccc;text-decoration:none;cursor:default}body{color:#222;font:normal 13px/1.4 arial",sans-serif;margin:0}.grecaptcha-badge{visibility:hidden}.uc-main{padding-top:50px;text-align:center}#uc-dl-icon{display:inline-block;margin-top:16px;padding-right:1em;vertical-align:top}#uc-text{display:inline-block;max-width:68ex;text-align:left}.uc-error-caption,".uc-warning-caption{color:#222;font-size:16px}#uc-download-link{text-decoration:none}.uc-name-size a{color:#15c;text-decoration:none}.uc-name-size a:visited{color:#61c;text-decoration:none}.uc-name-size a:active{color:#d14836;text-decoration:none}.uc-footer{color:#777;font-size:11px;padding-bottom:5ex;padding-top:5ex;text-align:center}.uc-footer a{color:#15c}.uc-footer a:visited{color:#61c}.uc-footer a:active{color:#d14836}.uc-footer-divider{color:#ccc;width:100%}.goog-inline-block{position:relative;display:-moz-inline-box;display:inline-block}* html .goog-inline-block{display:inline}:first-child+html .goog-inline-block{display:inline}sentinel{}</style><link rel=""icon"" href=""//ssl.gstatic.com/docs/doclist/images/drive_favicon_2026_32dp.png""/></head><body><div class=""uc-main""><div id=""uc-dl-icon"" class=""image-container""><div class=""drive-sprite-aux-download-file""></div></div><div id=""uc-text""><p class=""uc-warning-caption"">Google Drive can't scan this file for viruses.</p><p class=""uc-warning-subcaption""><span class=""uc-name-size""><a href=""/open?id=1x9151g2ZndSt87zZ03MEdBd6ak3rY5aG"">application_train.csv</a> (158M)</span> is too large for Google to scan for viruses. Would you still like to download this file?</p><form id=""download-form"" action=""https://drive.usercontent.google.com/download"" method=""get""><input type=""submit"" id=""uc-download-link"" class=""goog-inline-block jfk-button jfk-button-action"" value=""Download anyway""/><input type=""hidden"" name=""id"" value=""1x9151g2ZndSt87zZ03MEdBd6ak3rY5aG""><input type=""hidden"" name=""export"" value=""download""><input type=""hidden"" name=""confirm"" value=""t""><input type=""hidden"" name=""uuid"" value=""79faa669-e8af-463f-8683-474e69f2a098""></form></div></div><div class=""uc-footer""><hr class=""uc-footer-divider""></div></body></html>"
count,0,0,0
unique,0,0,0
top,NaN,NaN,NaN
freq,NaN,NaN,NaN


In [8]:
# Conteo del tipo de dato agrupado en columnas
conteo_tipos = HomeCredit_CD.dtypes.value_counts().reset_index()
conteo_tipos.columns = ['Tipo de Dato', 'Cantidad de Columnas']

display(conteo_tipos)

,Tipo de Dato,Cantidad de Columnas
0,object,3


Con el objetivo de comprender el dataset, se visualizan
 las variables con tipo de dato "object"

In [9]:
# Filtrar solo las columnas categóricas (objeto)
cols_object = HomeCredit_CD.select_dtypes(include=['object'])

# Ver la cantidad de columnas objeto y sus primeras filas
print(f"Total de columnas categóricas: {cols_object.shape[1]}\n")
display(cols_object.head())

Total de columnas categóricas: 3



,"<!DOCTYPE html><html><head><title>Google Drive - Virus scan warning</title><meta http-equiv=""content-type"" content=""text/html; charset=utf-8""/><style nonce=""0DJv_iFzrWvECY2jJuN1Gg"">.goog-link-button{position:relative;color:#15c;text-decoration:underline;cursor:pointer}.goog-link-button-disabled{color:#ccc;text-decoration:none;cursor:default}body{color:#222;font:normal 13px/1.4 arial",sans-serif;margin:0}.grecaptcha-badge{visibility:hidden}.uc-main{padding-top:50px;text-align:center}#uc-dl-icon{display:inline-block;margin-top:16px;padding-right:1em;vertical-align:top}#uc-text{display:inline-block;max-width:68ex;text-align:left}.uc-error-caption,".uc-warning-caption{color:#222;font-size:16px}#uc-download-link{text-decoration:none}.uc-name-size a{color:#15c;text-decoration:none}.uc-name-size a:visited{color:#61c;text-decoration:none}.uc-name-size a:active{color:#d14836;text-decoration:none}.uc-footer{color:#777;font-size:11px;padding-bottom:5ex;padding-top:5ex;text-align:center}.uc-footer a{color:#15c}.uc-footer a:visited{color:#61c}.uc-footer a:active{color:#d14836}.uc-footer-divider{color:#ccc;width:100%}.goog-inline-block{position:relative;display:-moz-inline-box;display:inline-block}* html .goog-inline-block{display:inline}:first-child+html .goog-inline-block{display:inline}sentinel{}</style><link rel=""icon"" href=""//ssl.gstatic.com/docs/doclist/images/drive_favicon_2026_32dp.png""/></head><body><div class=""uc-main""><div id=""uc-dl-icon"" class=""image-container""><div class=""drive-sprite-aux-download-file""></div></div><div id=""uc-text""><p class=""uc-warning-caption"">Google Drive can't scan this file for viruses.</p><p class=""uc-warning-subcaption""><span class=""uc-name-size""><a href=""/open?id=1x9151g2ZndSt87zZ03MEdBd6ak3rY5aG"">application_train.csv</a> (158M)</span> is too large for Google to scan for viruses. Would you still like to download this file?</p><form id=""download-form"" action=""https://drive.usercontent.google.com/download"" method=""get""><input type=""submit"" id=""uc-download-link"" class=""goog-inline-block jfk-button jfk-button-action"" value=""Download anyway""/><input type=""hidden"" name=""id"" value=""1x9151g2ZndSt87zZ03MEdBd6ak3rY5aG""><input type=""hidden"" name=""export"" value=""download""><input type=""hidden"" name=""confirm"" value=""t""><input type=""hidden"" name=""uuid"" value=""79faa669-e8af-463f-8683-474e69f2a098""></form></div></div><div class=""uc-footer""><hr class=""uc-footer-divider""></div></body></html>"


In [10]:
# Analisis de los valores únicos que posee cada columna categórica
HomeCredit_CD.select_dtypes(include=['object']).nunique()

,0
"<!DOCTYPE html><html><head><title>Google Drive - Virus scan warning</title><meta http-equiv=""content-type"" content=""text/html; charset=utf-8""/><style nonce=""0DJv_iFzrWvECY2jJuN1Gg"">.goog-link-button{position:relative;color:#15c;text-decoration:underline;cursor:pointer}.goog-link-button-disabled{color:#ccc;text-decoration:none;cursor:default}body{color:#222;font:normal 13px/1.4 arial",0
sans-serif;margin:0}.grecaptcha-badge{visibility:hidden}.uc-main{padding-top:50px;text-align:center}#uc-dl-icon{display:inline-block;margin-top:16px;padding-right:1em;vertical-align:top}#uc-text{display:inline-block;max-width:68ex;text-align:left}.uc-error-caption,0
".uc-warning-caption{color:#222;font-size:16px}#uc-download-link{text-decoration:none}.uc-name-size a{color:#15c;text-decoration:none}.uc-name-size a:visited{color:#61c;text-decoration:none}.uc-name-size a:active{color:#d14836;text-decoration:none}.uc-footer{color:#777;font-size:11px;padding-bottom:5ex;padding-top:5ex;text-align:center}.uc-footer a{color:#15c}.uc-footer a:visited{color:#61c}.uc-footer a:active{color:#d14836}.uc-footer-divider{color:#ccc;width:100%}.goog-inline-block{position:relative;display:-moz-inline-box;display:inline-block}* html .goog-inline-block{display:inline}:first-child+html .goog-inline-block{display:inline}sentinel{}</style><link rel=""icon"" href=""//ssl.gstatic.com/docs/doclist/images/drive_favicon_2026_32dp.png""/></head><body><div class=""uc-main""><div id=""uc-dl-icon"" class=""image-container""><div class=""drive-sprite-aux-download-file""></div></div><div id=""uc-text""><p class=""uc-warning-caption"">Google Drive can't scan this file for viruses.</p><p class=""uc-warning-subcaption""><span class=""uc-name-size""><a href=""/open?id=1x9151g2ZndSt87zZ03MEdBd6ak3rY5aG"">application_train.csv</a> (158M)</span> is too large for Google to scan for viruses. Would you still like to download this file?</p><form id=""download-form"" action=""https://drive.usercontent.google.com/download"" method=""get""><input type=""submit"" id=""uc-download-link"" class=""goog-inline-block jfk-button jfk-button-action"" value=""Download anyway""/><input type=""hidden"" name=""id"" value=""1x9151g2ZndSt87zZ03MEdBd6ak3rY5aG""><input type=""hidden"" name=""export"" value=""download""><input type=""hidden"" name=""confirm"" value=""t""><input type=""hidden"" name=""uuid"" value=""79faa669-e8af-463f-8683-474e69f2a098""></form></div></div><div class=""uc-footer""><hr class=""uc-footer-divider""></div></body></html>",0


### Valores Nulos del dataset

In [11]:
# 1. Calcular nulos totales y su porcentaje por columna
valores_nulos_totales = HomeCredit_CD.isnull().sum()
porcentaje_nulos = (valores_nulos_totales / len(HomeCredit_CD)) * 100

# 2. Unir ambos datos en un DataFrame ordenado
tabla_nulos = pd.DataFrame({
    'Valores Nulos': valores_nulos_totales,
    'Porcentaje (%)': porcentaje_nulos.round(2)
})

# 3. Filtrar solo las columnas que SI tienen nulos y ordenar de mayor a menor
tabla_nulos = tabla_nulos[tabla_nulos['Valores Nulos'] > 0].sort_values(by='Porcentaje (%)', ascending=False)

print(f"Total de columnas con valores nulos: {len(tabla_nulos)}")
display(tabla_nulos)

Total de columnas con valores nulos: 0


,Valores Nulos,Porcentaje (%)


## **Criterio de Tratamiento para Variables con Alto Porcentaje de Datos Faltantes**

Al analizar la distribución de los valores nulos en el dataset, se observa que una gran cantidad de columnas superan el 40% y 50% de datos ausentes (alcanzando en varios casos casi un 70%).

Ante un volumen de datos faltantes tan elevado, la estrategia metodológica que tomamos para tratar dichas variables es el de la eliminacion de las mismas en lugar de aplicar técnicas de imputación (relleno por media o mediana) por los siguientes motivos técnicos y estadísticos:


*   **Introducción de Sesgos Severos:** Imputar artificialmente más del 40% de los registros en una columna altera de forma sustancial la varianza, la desviación estándar y la distribución original de la variable, distorsionando la realidad de los datos.

*   **Inyección de Ruido al Modelo:** Rellenar valores masivamente con supuestos genera patrones ficticios que no aportan información predictiva real y pueden perjudicar el desempeño y la capacidad de generalización de los algoritmos de Machine Learning.


*   **Pérdida de Representatividad:** Cuando la ausencia de información es tan masiva, la variable pierde su utilidad analítica para caracterizar la población de clientes.


### **Estrategia a aplicar:**

Se mantendrán únicamente aquellas variables cuyo porcentaje de nulos sea inferior a un umbral operativo (ej. < 50%), reservando la imputación estadística exclusivamente para variables clave con porcentajes de ausencia marginales (< 5%).

In [12]:
#  Identificar las columnas a conservar(menos del 50% de nulos)
columnas_validas = tabla_nulos[tabla_nulos['Porcentaje (%)'] < 50].index

# Incluir las columnas que no tenían ningún nulo
columnas_sin_nulos = HomeCredit_CD.columns.difference(tabla_nulos.index)
todas_las_columnas_a_conservar = columnas_sin_nulos.union(columnas_validas)

#  Crear un nuevo dataset reducido por columnas (Se crea un nuevo Dataframe 'df_reducido' para trabajar la transformación.
#  Esto evita la mutación de datos y conserva 'HomeCredit_CD' como fuente intacta)
df_reducido = HomeCredit_CD[todas_las_columnas_a_conservar].copy()

#  Comprobar dimensiones
print(f"Dataset original (HomeCredit_CD): {HomeCredit_CD.shape[1]} columnas")
print(f"Nuevo dataset (df_reducido): {df_reducido.shape[1]} columnas")
print(f"Columnas descartadas por exceso de nulos: {HomeCredit_CD.shape[1] - df_reducido.shape[1]}")

Dataset original (HomeCredit_CD): 3 columnas
Nuevo dataset (df_reducido): 3 columnas
Columnas descartadas por exceso de nulos: 0


# **Analisis de las 41 columnas con una cantidad de datos nulos superior al 50%**

In [13]:
#Identificar las 41 columnas con más del 50% de nulos
porcentaje_nulos = (HomeCredit_CD.isnull().sum() / len(HomeCredit_CD)) * 100
columnas_eliminadas = porcentaje_nulos[porcentaje_nulos > 50].index.tolist()

#Crear el dataset reducido (81 columnas)
df_reducido = HomeCredit_CD.drop(columns=columnas_eliminadas).copy()

#Mostrar la lista exacta de las columnas eliminadas
print(f"Total de columnas eliminadas: {len(columnas_eliminadas)}\n")
print(columnas_eliminadas)


Total de columnas eliminadas: 0

[]


### **Listado Detallado de las 41 Columnas Eliminadas (> 50% de Nulos)**

Se descartaron exactamente **41 variables** por superar el 50% de datos faltantes para evitar sesgar el análisis mediante imputaciones artificiales:

| # | Nombre de la Columna | Categoría | Descripción |
| :-: | :--- | :--- | :--- |
| **1** | `OWN_CAR_AGE` | Vehículo | Antigüedad (en años) del vehículo del cliente. |
| **2** | `EXT_SOURCE_1` | Scoring Externo | Puntuación normalizada proveniente de la fuente externa #1. |
| **3** | `APARTMENTS_AVG` | Vivienda (Promedio) | Tamaño del departamento normalizado. |
| **4** | `APARTMENTS_MODE` | Vivienda (Moda) | Tamaño del departamento normalizado. |
| **5** | `APARTMENTS_MEDI` | Vivienda (Mediana) | Tamaño del departamento normalizado. |
| **6** | `BASEMENTAREA_AVG` | Vivienda (Promedio) | Área del sótano normalizada. |
| **7** | `BASEMENTAREA_MODE` | Vivienda (Moda) | Área del sótano normalizada. |
| **8** | `BASEMENTAREA_MEDI` | Vivienda (Mediana) | Área del sótano normalizada. |
| **9** | `YEARS_BUILD_AVG` | Vivienda (Promedio) | Antigüedad / año de construcción del edificio. |
| **10** | `YEARS_BUILD_MODE` | Vivienda (Moda) | Antigüedad / año de construcción del edificio. |
| **11** | `YEARS_BUILD_MEDI` | Vivienda (Mediana) | Antigüedad / año de construcción del edificio. |
| **12** | `COMMONAREA_AVG` | Vivienda (Promedio) | Área común del inmueble normalizada. |
| **13** | `COMMONAREA_MODE` | Vivienda (Moda) | Área común del inmueble normalizada. |
| **14** | `COMMONAREA_MEDI` | Vivienda (Mediana) | Área común del inmueble normalizada. |
| **15** | `ELEVATORS_AVG` | Vivienda (Promedio) | Cantidad de ascensores en el edificio. |
| **16** | `ELEVATORS_MODE` | Vivienda (Moda) | Cantidad de ascensores en el edificio. |
| **17** | `ELEVATORS_MEDI` | Vivienda (Mediana) | Cantidad de ascensores en el edificio. |
| **18** | `ENTRANCES_AVG` | Vivienda (Promedio) | Cantidad de entradas al edificio. |
| **19** | `ENTRANCES_MODE` | Vivienda (Moda) | Cantidad de entradas al edificio. |
| **20** | `ENTRANCES_MEDI` | Vivienda (Mediana) | Cantidad de entradas al edificio. |
| **21** | `FLOORSMAX_AVG` | Vivienda (Promedio) | Número máximo de pisos en el edificio. |
| **22** | `FLOORSMIN_AVG` | Vivienda (Promedio) | Número mínimo de pisos en el edificio. |
| **23** | `FLOORSMIN_MODE` | Vivienda (Moda) | Número mínimo de pisos en el edificio. |
| **24** | `FLOORSMIN_MEDI` | Vivienda (Mediana) | Número mínimo de pisos en el edificio. |
| **25** | `LANDAREA_AVG` | Vivienda (Promedio) | Área del terreno normalizada. |
| **26** | `LANDAREA_MODE` | Vivienda (Moda) | Área del terreno normalizada. |
| **27** | `LANDAREA_MEDI` | Vivienda (Mediana) | Área del terreno normalizada. |
| **28** | `LIVINGAPARTMENTS_AVG` | Vivienda (Promedio) | Área departamental habitable normalizada. |
| **29** | `LIVINGAPARTMENTS_MODE` | Vivienda (Moda) | Área departamental habitable normalizada. |
| **30** | `LIVINGAPARTMENTS_MEDI` | Vivienda (Mediana) | Área departamental habitable normalizada. |
| **31** | `LIVINGAREA_AVG` | Vivienda (Promedio) | Área habitable total normalizada. |
| **32** | `LIVINGAREA_MODE` | Vivienda (Moda) | Área habitable total normalizada. |
| **33** | `LIVINGAREA_MEDI` | Vivienda (Mediana) | Área habitable total normalizada. |
| **34** | `NONLIVINGAPARTMENTS_AVG` | Vivienda (Promedio) | Área departamental no habitable normalizada. |
| **35** | `NONLIVINGAPARTMENTS_MODE` | Vivienda (Moda) | Área departamental no habitable normalizada. |
| **36** | `NONLIVINGAPARTMENTS_MEDI` | Vivienda (Mediana) | Área departamental no habitable normalizada. |
| **37** | `NONLIVINGAREA_AVG` | Vivienda (Promedio) | Área no habitable total normalizada. |
| **38** | `NONLIVINGAREA_MODE` | Vivienda (Moda) | Área no habitable total normalizada. |
| **39** | `NONLIVINGAREA_MEDI` | Vivienda (Mediana) | Área no habitable total normalizada. |
| **40** | `HOUSETYPE_MODE` | Vivienda (Moda) | Tipo de vivienda u hogar. |
| **41** | `WALLSMATERIAL_MODE` | Vivienda (Moda) | Material de construcción de las paredes externas. |

---

> **Conclusión técnica:**
> * **39 variables de infraestructura:** Corresponden a detalles arquitectónicos de la vivienda que la mayoría de los clientes no declararon.
> * **`OWN_CAR_AGE`:** Presente únicamente en clientes que declararon poseer vehículo propio.
> * **`EXT_SOURCE_1`:** Score crediticio externo ausente en más del 50% de la muestra.

# **Estrategia de Categorización y Tratamiento de Valores Nulos Restantes**

Tras haber removido las columnas con ausencia masiva (> 50%), el dataset `df_reducido` aún presenta variables con datos faltantes. Debido a que la cantidad de nulos difiere sustancialmente entre columnas, **no resulta conveniente aplicar una única técnica de imputación global**, ya que esto podría sesgar la distribución de los datos o distorsionar patrones predictivos clave.

Para abordar este proceso de manera estratégica y metodológicamente rigurosa, se clasificaron las variables con nulos en tres franjas según su porcentaje de ausencia:

* **1. Nivel Alto (30% a 50%):** Reemplazar las ausencias por la etiqueta `Desconocido` en variables categóricas. En variables numéricas de este rango VER QUE HACER (ANALIZAR CASOS AISLADOS).
* **2. Nivel Medio (5% a 30%):** Se aplica imputación por **mediana**  para preservar el alto poder predictivo de estas métricas.
* **3. Nivel Bajo (< 5%):**  Al representar una pérdida de información mínima, el riesgo de distorsión estadística es bajo. Se aplica imputación directa por **mediana** para variables continuas y por la **moda** (valor más frecuente) para variables categóricas.

Complementariamente al criterio cuantitativo aplicado por franjas de vacíos, se llevará a cabo un análisis cualitativo e individual por variable. El objetivo es evaluar su relevancia del negocio y capacidad predictiva respecto al fenómeno en estudio. De este modo, la decisión final de inclusión o descarte no dependerá únicamente del volumen de faltantes; variables con baja tasa de ausencias podrían ser omitidas por redundancia o falta de poder explicativo, mientras que variables pertenecientes a franjas de alta vaciedad podrían ser retenidas e imputadas mediante técnicas avanzadas si aportan valor crítico al modelo.


---


**Objetivo del Enfoque:**
Esta segmentación por niveles asegura un tratamiento diferenciado y adaptado al volumen de datos faltantes, preservando la calidad estadística del dataset antes de proceder con el modelado.

In [14]:
#Calcular nulos y porcentajes únicamente sobre df_reducido
nulos = df_reducido.isnull().sum()
porcentajes = (nulos / len(df_reducido)) * 100

#Filtrar solo las columnas que conservan nulos (> 0%)
df_nulos = pd.DataFrame(
    {
        'Columna': nulos.index,
        'Tipo_Dato': df_reducido.dtypes.values,
        'Valores_Nulos': nulos.values,
        'Porcentaje (%)': porcentajes.round(2).values,
    }
)
df_nulos = df_nulos[df_nulos['Valores_Nulos'] > 0]


#Función para clasificar en las 3 franjas de criticidad
def clasificar_franja(pct):
    if pct >= 30:
        return '1. Alto (30% - 50%)'
    elif pct >= 5:
        return '2. Medio (5% - 30%)'
    else:
        return '3. Bajo (< 5%)'


df_nulos['Franja'] = df_nulos['Porcentaje (%)'].apply(clasificar_franja)

#Ordenar y mostrar la tabla agrupada
tabla_clasificada = df_nulos.sort_values(
    by=['Franja', 'Porcentaje (%)'], ascending=[True, False]
)

display(tabla_clasificada.reset_index(drop=True))



,Columna,Tipo_Dato,Valores_Nulos,Porcentaje (%),Franja


### **Resumen General de la Distribución de Variables (122 Columnas Totales)**

El total de **122 variables** del dataset original se clasificó de la siguiente manera:

* **Columnas Eliminadas (41 variables):** Superaban el 50% de valores faltantes.
* **Dataset Reducido / Conservado (81 variables):**
  * **Sin nulos (55 columnas):** Tienen el **100% de sus datos completos**.
  * **Con nulos (26 columnas):** Variables que conservan faltantes y serán imputadas:
    * **Franja Alta (30% - 50%):** 9 columnas
    * **Franja Media (5% - 30%):** 7 columnas
    * **Franja Baja (< 5%):** 10 columnas

---

### **Desglose de las 26 Variables a Imputar por Franja**

#### **1. Franja Alta (30% a 50% de Nulos - 9 columnas)**

| Nombre de la Columna | Tipo de Dato | % Nulos | Descripción |
| :--- | :--- | :-: | :--- |
| **`FLOORSMAX_AVG`** | Numérico | 49.76% | Número máximo de pisos en el edificio (promedio). |
| **`FLOORSMAX_MODE`** | Numérico | 49.76% | Número máximo de pisos en el edificio (moda). |
| **`FLOORSMAX_MEDI`** | Numérico | 49.76% | Número máximo de pisos en el edificio (mediana). |
| **`YEARS_BEGINEXPLUATATION_AVG`** | Numérico | 48.78% | Antigüedad/inicio de explotación del edificio (promedio). |
| **`YEARS_BEGINEXPLUATATION_MODE`** | Numérico | 48.78% | Antigüedad/inicio de explotación del edificio (moda). |
| **`YEARS_BEGINEXPLUATATION_MEDI`** | Numérico | 48.78% | Antigüedad/inicio de explotación del edificio (mediana). |
| **`TOTALAREA_MODE`** | Numérico | 48.27% | Área total del inmueble normalizada (moda). |
| **`EMERGENCYSTATE_MODE`** | Categórico | 47.40% | Indicador si la vivienda presenta estado de emergencia/riesgo. |
| **`OCCUPATION_TYPE`** | Categórico | 31.35% | Tipo de ocupación, profesión u oficio que desempeña el cliente. |

---

#### **2. Franja Media (5% a 30% de Nulos - 7 columnas)**

| Nombre de la Columna | Tipo de Dato | % Nulos | Descripción |
| :--- | :--- | :-: | :--- |
| **`EXT_SOURCE_3`** | Numérico | 19.83% | Puntuación/score de riesgo normalizado de la fuente externa #3. |
| **`AMT_REQ_CREDIT_BUREAU_HOUR`** | Numérico | 13.50% | Consultas al Bureau de Crédito 1 hora antes de la solicitud. |
| **`AMT_REQ_CREDIT_BUREAU_DAY`** | Numérico | 13.50% | Consultas al Bureau de Crédito 1 día antes de la solicitud. |
| **`AMT_REQ_CREDIT_BUREAU_WEEK`** | Numérico | 13.50% | Consultas al Bureau de Crédito 1 semana antes de la solicitud. |
| **`AMT_REQ_CREDIT_BUREAU_MON`** | Numérico | 13.50% | Consultas al Bureau de Crédito 1 mes antes de la solicitud. |
| **`AMT_REQ_CREDIT_BUREAU_QRT`** | Numérico | 13.50% | Consultas al Bureau de Crédito 3 meses antes de la solicitud. |
| **`AMT_REQ_CREDIT_BUREAU_YEAR`** | Numérico | 13.50% | Consultas al Bureau de Crédito 1 año antes de la solicitud. |

---

#### **3. Franja Baja (< 5% de Nulos - 10 columnas)**

| Nombre de la Columna | Tipo de Dato | % Nulos | Descripción |
| :--- | :--- | :-: | :--- |
| **`NAME_TYPE_SUITE`** | Categórico | 0.42% | Acompañante presente al solicitar el préstamo. |
| **`OBS_30_CNT_SOCIAL_CIRCLE`** | Numérico | 0.33% | Personas del entorno social observadas con atrasos de 30 días. |
| **`DEF_30_CNT_SOCIAL_CIRCLE`** | Numérico | 0.33% | Personas del entorno social que cayeron en mora a 30 días. |
| **`OBS_60_CNT_SOCIAL_CIRCLE`** | Numérico | 0.33% | Personas del entorno social observadas con atrasos de 60 días. |
| **`DEF_60_CNT_SOCIAL_CIRCLE`** | Numérico | 0.33% | Personas del entorno social que cayeron en mora a 60 días. |
| **`EXT_SOURCE_2`** | Numérico | 0.21% | Puntuación/score de riesgo normalizado de la fuente externa #2. |
| **`AMT_GOODS_PRICE`** | Numérico | 0.09% | Precio del bien o servicio para el cual se otorga el crédito. |
| **`AMT_ANNUITY`** | Numérico | <0.01% | Monto de la cuota/anualidad fija a pagar por el préstamo. |
| **`CNT_FAM_MEMBERS`** | Numérico | <0.01% | Cantidad total de integrantes dentro del grupo familiar. |
| **`DAYS_LAST_PHONE_CHANGE`** | Numérico | <0.01% | Días transcurridos desde que el cliente cambió su teléfono. |